# 6.3 Imaging And Inversion-Ready Outputs

This tutorial introduces FrequenSolve imaging jobs and the image-output reader. It builds a small elastic acquisition, defines an RTM-style imaging job, inspects the exported solver contract, previews the image database format, and shows the strict run path for generating observed frequency-domain data and imaging it.

The purpose is practical: after this notebook, you should know which Python object owns each imaging decision, how observed and simulated traces are paired, how FWI-gradient and field images are requested, and how solver image files are opened as `xarray` datasets for plotting or downstream inversion workflows.

## How To Read This Tutorial

Imaging is where FrequenSolve starts to look less like a conventional forward-modeling wrapper and more like an inversion platform. The same project/simulation/job structure is used, but the job now connects observed data, simulated data, image conditions, frequency weights, and image grids.

Read this notebook in two layers: first as an authoring tutorial for imaging contracts, then as a strict solver workflow that generates observed synthetic data and runs an imaging job.

## Imaging Vocabulary

Imaging in FrequenSolve is a job workflow attached to a simulation. The forward simulation defines the background model, mesh, acquisition, source wavelet, and receiver components. The imaging job adds an observed-data path, a frequency list, an image grid, a misfit definition, and a list of image conditions.

| Concept | Python API | What it controls |
| --- | --- | --- |
| Background simulation | `project.new_simulation(..., physics="elastic")` | The model and acquisition used to compute simulated data and adjoint wavefields. |
| Observed data | `observed=...` in `sim.imaging_job(...)` | Frequency-domain trace data to compare against the simulated traces. This can be a trace-producing job, a `TraceDataset`, or a filesystem path. |
| Misfit | `misfit_norm="L2"` or `misfit_type="L2"` | How residual traces are measured. The current public tutorial path uses `L2`. |
| Image grid | `fs.CartesianGrid(...)` or a resolution sequence | The regular grid where image volumes and FWI gradients are accumulated. |
| FWI-gradient images | `parameters=["vp", "vs", "rho"]` | Requests image conditions serialized as `FWI:Vp`, `FWI:Vs`, and `FWI:Rho`. |
| Field/condition images | `fields=[...]`, `condition=...`, or `images={...}` | Requests diagnostic wavefield images such as `velocity`, `pressure`, or named solver image conditions. |
| Image reader | `fs.ImageDatabase(...)` or `site.fetch_image(job)` | Opens solver `image.h5` output as `xarray.Dataset` objects.

## Data Pairing Is The Central Contract

An imaging job is meaningful only if observed and simulated receiver groups are paired correctly. The observed data may come from field traces, a previously run synthetic job, or a fetched remote result; the simulated side comes from the migration/background simulation.

Before a large imaging run, inspect the exported `Image` block and confirm the observed path, receiver-group names, components, frequency list, weights, and image grid. Most imaging mistakes are contract mistakes before they are numerical mistakes.

## Imports And Project Path

The notebook writes a small scratch project under `./scratch/tutorials/imaging`. The image preview section also writes a tiny synthetic HDF5 image store with the same layout expected by `ImageDatabase`; that preview is for teaching the reader API and plotting workflow before running the solver.

In [ ]:
from pathlib import Path
from pprint import pprint

import h5py
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import frequensolve as fs

u = fs.ureg

In [ ]:
project = fs.Project(
    name="project",
    pretty_name="Imaging Tutorial",
    path="./scratch/tutorials/imaging",
    log_level="INFO",
    log_to_console=True,
)

project.path

## Build Matched Smooth And True Simulations

A common imaging workflow uses two models:

- a **true** or observed-data model, which stands in for field data or a higher-fidelity synthetic experiment;
- a **smooth** background model, which is used for migration and gradient computation.

For a compact tutorial, both simulations use the same flat two-layer elastic property values from the Modeling Basics notebooks. The only deliberate mismatch is the interface depth: the true model has a slightly deeper interface. When the solver is run, that mismatch creates residuals concentrated around the reflector, which is exactly the kind of structure an imaging condition should expose.

In [ ]:
def build_elastic_simulation(project, *, name, interface_depth_km):
    sim = project.new_simulation(
        name=name,
        physics="elastic",
        dimension=2,
        units={
            "length": "km",
            "velocity": "km/s",
            "density": "g/cm^3",
        },
    )

    model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.2])
    model.add_surface(name="top", depth=0.0 * u.km)
    model.add_layer(
        name="upper_layer",
        properties={
            "Vp": 2.0 * u.km / u.s,
            "Vs": 1.0 * u.km / u.s,
            "Rho": 2.2 * u.g / u.cm**3,
        },
    )
    model.add_surface(name="interface", depth=interface_depth_km * u.km)
    model.add_layer(
        name="lower_layer",
        properties={
            "Vp": 2.8 * u.km / u.s,
            "Vs": 1.5 * u.km / u.s,
            "Rho": 2.4 * u.g / u.cm**3,
        },
    )
    model.add_surface(name="bottom", depth=0.5 * u.km)
    sim += model

    sim += model.hex_mesh_generator([12, 5])
    sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=25.0)
    sim.mesh.set_source_grading(d1=0.05, factor=2.0)

    sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
    sim += fs.BoundaryCondition(
        conditions=["pml"],
        boundaries=["x_min", "x_max", "z_max"],
        pml_wavelengths=0.75,
    )

    acq = fs.Acquisition()
    acq.add_sources(
        kind="vector",
        coords=fs.Q_([[0.25, 0.02], [0.60, 0.02], [0.95, 0.02]], "km"),
        direction=[0.0, 1.0],
    )

    geophone = fs.ReceiverNode(name="surface_geophone")
    geophone.add_component(name="v_z", field="velocity", direction=[0.0, 1.0])

    receiver_coords = [fs.Q_([x, 0.0], "km") for x in np.linspace(0.05, 1.15, 121)]
    acq.add_receiver_group(name="surface", device=geophone, coords=receiver_coords)
    sim += acq

    sim += fs.SolverConfig(ptol=1.0e-10)
    return sim


smooth_sim = build_elastic_simulation(
    project,
    name="imaging_smooth",
    interface_depth_km=0.25,
)
true_sim = build_elastic_simulation(
    project,
    name="imaging_true",
    interface_depth_km=0.30,
)

## Inspect The Background And True Models

The plots below are intentionally simple. The background model is what the imaging job uses for simulated data and adjoint propagation. The true model is only used here to synthesize observed data. In field-data workflows, `true_sim` would be replaced by a path to measured or preprocessed frequency-domain traces.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True)
smooth_sim.model.plot("vp", ax=axes[0], aspect="equal")
axes[0].set_title("Smooth migration model: Vp")
true_sim.model.plot("vp", ax=axes[1], aspect="equal")
axes[1].set_title("True observed-data model: Vp");

## Frequencies, Weights, And The Image Grid

Frequency-domain imaging works on a selected set of frequencies. A few low-to-mid frequencies are enough for this small tutorial model and keep the job contract readable. `weights` may be supplied explicitly, or a wavelet can be supplied so the imaging job can derive frequency weights from the wavelet spectrum.

The image grid is independent of the finite-element mesh. It is a regular `CartesianGrid` where image values are accumulated and later read into `xarray`. In 2D, `x0` and `x1` are `[x, z]`; the resulting `xarray` image arrays are stored with dimensions `(z, x)` so image plotting follows the usual depth-versus-distance layout.

In [ ]:
frequencies = [8.0, 12.0, 18.0]
frequency_weights = [1.0, 0.8, 0.45]

image_grid = fs.CartesianGrid(
    n=[161, 81],
    x0=[0.0, 0.0],
    x1=[1.2, 0.5],
)

image_grid.as_xarray()

## Author An Imaging Job

`sim.imaging_job(...)` is the high-level public constructor. It normalizes property names, expands `parameters=["vp", "vs", "rho"]` into solver FWI-gradient image requests, and pairs observed and simulated receiver groups by name.

For this authoring cell, the observed path is a placeholder directory so the job can be constructed, saved, loaded, and inspected without first running the solver. The strict run section later rebuilds the imaging job against a real trace-producing `FrequencyDomainJob`.

In [ ]:
observed_root = Path(project.path) / "observed_frequency_data"
(observed_root / "surface").mkdir(parents=True, exist_ok=True)

imaging_job = smooth_sim.imaging_job(
    name="rtm_elastic",
    observed=observed_root,
    frequencies=frequencies,
    grid=image_grid,
    parameters=["vp", "vs", "rho"],
    fields=["velocity"],
    condition="up_down",
    weights=frequency_weights,
    misfit_norm="L2",
    keep_forward=False,
    keep_adjoint=False,
    keep_unstacked=False,
)

imaging_job_file = imaging_job.save()
loaded_imaging_job = fs.BaseJob.load(imaging_job_file)

imaging_job_file, type(loaded_imaging_job).__name__, loaded_imaging_job.images

## Read The Imaging Contract

The `Image` section is the part of the exported job consumed by the imaging workflow. The most important fields to check before submitting a large run are:

| Contract field | Why it matters |
| --- | --- |
| `data_path` | Root observed-data path. For receiver group `surface`, the job pairs `data_path/surface` with simulated traces. |
| `misfit.receiver_groups` | Shows each observed/simulated receiver-group pairing. Names must match the acquisition geometry. |
| `grid` | The regular output grid for image accumulation. |
| `images` | The requested imaging conditions and FWI-gradient properties. |
| `weights` | Frequency weights applied during stacking. |
| `keep_forward`, `keep_adjoint`, `keep_unstacked` | Debug/storage switches. Keep these off unless you need intermediate wavefields or per-frequency images.

In [ ]:
image_contract = imaging_job.to_fs()["Image"]
pprint(image_contract)

## Image Request Styles

The natural API is usually the clearest style:

```python
smooth_sim.imaging_job(parameters=["vp", "vs", "rho"], fields=["velocity"], condition="up_down", ...)
```

For advanced workflows, `images={...}` can pass exact image-condition names. Strings beginning with `FWI:` request a property-gradient image; other strings are passed as image-condition names. The two cells below build equivalent focused requests.

In [ ]:
focused_job = smooth_sim.imaging_job(
    name="rtm_focused",
    observed=observed_root,
    frequencies=[12.0],
    grid=image_grid,
    images={
        "dVp": "FWI:Vp",
        "dRho": "FWI:Rho",
        "vz_image": "velocity",
    },
    weights=[1.0],
)

focused_job.to_fs()["Image"]["images"]

## Preview ImageDatabase Output

When an imaging run succeeds, sites return an `ImageDatabase` through `site.fetch_image(imaging_job)`. The database reads solver HDF5 files and exposes two common datasets:

- `raw_images`: unfiltered images from the `image/raw` group;
- `smoothed_images`: smoothed images from the `image/smoothed` group.

The next cell writes a tiny solver-shaped image store and reads it with the same public `ImageDatabase` class. The data are synthetic, but the file layout and plotting code are the same pattern you use for solver output.

In [ ]:
def write_synthetic_image_database(path, *, grid, frequency):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)

    nx = int(grid.n[0])
    nz = int(grid.n[1])
    x = np.linspace(grid.x0[0], grid.x1[0], nx)[None, :]
    z = np.linspace(grid.x0[1], grid.x1[1], nz)[:, None]

    reflector = np.exp(-((z - 0.30) ** 2) / (2.0 * 0.015**2))
    aperture = np.cos(np.pi * (x - 0.60) / 1.2) ** 2
    dip = np.sin(2.0 * np.pi * (x / 1.2 + 0.8 * z))

    raw_vp = reflector * aperture * (1.0 + 0.25 * dip)
    raw_vs = 0.55 * reflector * aperture * (1.0 - 0.15 * dip)
    raw_rho = -0.35 * reflector * aperture

    smoothed = {
        "FWI_Vp": 0.72 * raw_vp,
        "FWI_Vs": 0.72 * raw_vs,
        "FWI_Rho": 0.72 * raw_rho,
    }
    raw = {"FWI_Vp": raw_vp, "FWI_Vs": raw_vs, "FWI_Rho": raw_rho}

    string_dtype = h5py.string_dtype(encoding="utf-8")
    with h5py.File(path / "image.h5", "w") as h5:
        for group_name, images in {"raw": raw, "smoothed": smoothed}.items():
            group = h5.create_group(f"image/{group_name}")
            group.create_dataset(
                "properties",
                data=np.array(list(images), dtype=string_dtype),
            )
            for name, values in images.items():
                dataset = group.create_dataset(name, data=values.reshape(-1))
                dataset.attrs["x0"] = np.array([grid.x0[0], grid.x0[1]])
                dataset.attrs["x1"] = np.array([grid.x1[0], grid.x1[1]])
                dataset.attrs["n_grid"] = np.array([nx, nz])
                dataset.attrs["dims"] = np.array(["x", "z"], dtype=string_dtype)

    with h5py.File(path / "image_1.h5", "w") as h5:
        h5.create_dataset("frequency", data=float(frequency))


preview_path = Path(project.path) / "synthetic_image_preview"
write_synthetic_image_database(preview_path, grid=image_grid, frequency=12.0)

image_db = fs.ImageDatabase(path=preview_path, parts=1, shape=image_grid.shape)
raw_images = image_db.raw_images
smoothed_images = image_db.smoothed_images

display(raw_images)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 5), sharex=True, sharey=True, constrained_layout=True)

for col, name in enumerate(["FWI_Vp", "FWI_Vs", "FWI_Rho"]):
    raw_images[name].plot.imshow(
        ax=axes[0, col],
        x="x",
        y="z",
        yincrease=False,
        cmap="RdBu_r",
        add_colorbar=True,
    )
    axes[0, col].set_title(f"raw {name}")

    smoothed_images[name].plot.imshow(
        ax=axes[1, col],
        x="x",
        y="z",
        yincrease=False,
        cmap="RdBu_r",
        add_colorbar=True,
    )
    axes[1, col].set_title(f"smoothed {name}")

for ax in axes.ravel():
    ax.set_xlabel("x [km]")
    ax.set_ylabel("z [km]")

fig.suptitle("ImageDatabase preview: raw and smoothed FWI-gradient images");

## Inversion-Ready Vector Spaces

The imaging API also exposes FWI-oriented model and data vectorization through `sim.fwi(...)`. This does not run an inversion by itself. It defines how model perturbations and residual traces are packed for external inversion algorithms or site implementations that provide Jacobian, adjoint, and nonlinear-forward hooks.

Use this when you need a PyLops-compatible operator interface. Use `sim.imaging_job(...)` when you want the solver to run an RTM/adjoint imaging job and write image files.

In [ ]:
fwi_problem = smooth_sim.fwi(
    observed=observed_root,
    frequencies=frequencies,
    parameters=["vp", "vs", "rho"],
    grid=image_grid,
)

summary = {
    "model_parameters": fwi_problem.model_space.parameters,
    "model_vector_size": fwi_problem.model_space.size,
    "data_vector_size": fwi_problem.data_space.size,
    "image_grid_dims": fwi_problem.model_space.dims,
    "image_grid_shape": fwi_problem.grid.shape,
}
summary

## Strict Solver Run: Generate Observed Data

The remaining cells are the real solver path. They are intentionally strict: if the configured site's solver or imaging workflow is unavailable, the notebook should fail here and leave logs, job JSON, and result paths for inspection.

First run a frequency-domain job on the true model to synthesize observed data. In a field-data workflow, this job is replaced by a path to preprocessed observed frequency-domain traces with receiver-group names matching the simulation acquisition.

In [ ]:
site = fs.Site()

observed_job = fs.FrequencyDomainJob(
    name="observed_true_data",
    simulation=true_sim,
    f_list=frequencies,
)

observed_result = site.submit(observed_job).wait()
observed_traces = observed_result.traces()
observed_traces.summary

## Strict Solver Run: Run The Imaging Job

Now rebuild the imaging job with `observed=observed_job`. Passing a trace-producing job lets FrequenSolve infer both the observed trace path and the frequency list. After the run succeeds, `site.fetch_image(...)` returns an `ImageDatabase`, and the plotting pattern from the preview section applies directly to solver output.

In [ ]:
rtm_job = smooth_sim.imaging_job(
    name="rtm_from_true_data",
    observed=observed_job,
    grid=image_grid,
    parameters=["vp", "vs", "rho"],
    fields=["velocity"],
    condition="up_down",
    weights=frequency_weights,
    misfit_norm="L2",
)

rtm_result = site.submit(rtm_job).wait()
solver_images = site.fetch_image(rtm_job)
solver_images.raw_images

In [ ]:
solver_raw = solver_images.raw_images
image_names = list(solver_raw.data_vars)

fig, axes = plt.subplots(1, len(image_names), figsize=(4 * len(image_names), 3), constrained_layout=True)
if len(image_names) == 1:
    axes = [axes]

for ax, name in zip(axes, image_names):
    solver_raw[name].plot.imshow(
        ax=ax,
        x="x",
        y="z",
        yincrease=False,
        cmap="RdBu_r",
        add_colorbar=True,
    )
    ax.set_title(name)
    ax.set_xlabel("x [km]")
    ax.set_ylabel("z [km]")

## Before Moving On

Before scaling imaging, inspect the contract more carefully than the final picture. Receiver-group pairings, observed-data paths, image-grid bounds, frequency weights, and requested image conditions determine whether the image is meaningful.

The preview section is intentionally synthetic so users can learn the `ImageDatabase` reader before running expensive jobs. The strict run section then connects that reader to real solver output.

## Review Checklist

Before scaling this workflow, inspect these artifacts:

| Artifact | What to check |
| --- | --- |
| `imaging_job.to_fs()["Image"]` | Observed/simulated receiver paths, requested image conditions, grid bounds, and frequency weights. |
| Saved imaging job JSON | The job can be saved and loaded like other FrequenSolve jobs. |
| Observed trace job results | Receiver group names and components match the migration simulation. |
| `ImageDatabase.raw_images` | FWI-gradient and diagnostic image names are present with expected `(z, x)` dimensions. |
| `ImageDatabase.smoothed_images` | Smoothed images are available when the solver writes the `smoothed` group. |
| Logs and result directory | Solver-side failures are debugged from the strict run artifacts, not hidden in the notebook. |